In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)


In [ ]:
df = pd.read_excel("Online Retail.xlsx")
df.head()


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.00,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.00,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.00,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.00,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.00,United Kingdom


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


In [ ]:
df.isna().sum()

,0
InvoiceNo,0
StockCode,0
Description,1454
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,135080
Country,0


In [ ]:
df.describe()


,Quantity,InvoiceDate,UnitPrice,CustomerID
count,541909.00,541909,541909.00,406829.00
mean,9.55,2011-07-04 13:34:57.156386048,4.61,15287.69
min,-80995.00,2010-12-01 08:26:00,-11062.06,12346.00
25%,1.00,2011-03-28 11:34:00,1.25,13953.00
50%,3.00,2011-07-19 17:17:00,2.08,15152.00
75%,10.00,2011-10-19 11:27:00,4.13,16791.00
max,80995.00,2011-12-09 12:50:00,38970.00,18287.00
std,218.08,NaN,96.76,1713.60


In [ ]:
df_raw = df.copy()
df = df.copy()


In [ ]:
df.columns = [c.strip() for c in df.columns]

df["InvoiceNo"] = df["InvoiceNo"].astype(str).str.strip()
df["StockCode"] = df["StockCode"].astype(str).str.strip()
df["Description"] = df["Description"].astype(str).str.strip()
df["Country"] = df["Country"].astype(str).str.strip()

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["CustomerID"] = pd.to_numeric(df["CustomerID"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")


In [ ]:
df["is_cancelled"] = df["InvoiceNo"].str.startswith("C")


In [ ]:
gross = df.dropna(subset=["InvoiceDate"]).copy()
gross = gross[(gross["UnitPrice"] > 0) & (gross["Quantity"] != 0)]
gross["gross_revenue"] = gross["Quantity"] * gross["UnitPrice"]


In [ ]:
net = gross[~gross["is_cancelled"]].copy()
net = net[net["Quantity"] > 0]
net["net_revenue"] = net["Quantity"] * net["UnitPrice"]


In [ ]:
for dfx in [gross, net]:
    dfx["invoice_date"] = dfx["InvoiceDate"].dt.date
    dfx["year"] = dfx["InvoiceDate"].dt.year
    dfx["month"] = dfx["InvoiceDate"].dt.month
    dfx["day"] = dfx["InvoiceDate"].dt.day
    dfx["year_month"] = dfx["InvoiceDate"].dt.to_period("M").astype(str)


In [ ]:
summary = pd.DataFrame({
    "dataset": ["raw", "gross_clean", "net_clean"],
    "rows": [len(df_raw), len(gross), len(net)],
    "unique_invoices": [
        df_raw["InvoiceNo"].nunique(),
        gross["InvoiceNo"].nunique(),
        net["InvoiceNo"].nunique()
    ],
    "unique_customers": [
        df_raw["CustomerID"].nunique(),
        gross["CustomerID"].nunique(),
        net["CustomerID"].nunique()
    ],
})
summary


,dataset,rows,unique_invoices,unique_customers
0,raw,541909,25900,4372
1,gross_clean,539392,23796,4371
2,net_clean,530104,19960,4338


In [ ]:
sales = net.copy()
sales.shape


(530104, 16)

In [ ]:
dates = pd.to_datetime(sales["InvoiceDate"]).dt.date
dim_date = pd.DataFrame({"full_date": pd.to_datetime(sorted(dates.unique()))})

dim_date["date_key"] = dim_date["full_date"].dt.strftime("%Y%m%d").astype(int)
dim_date["year"] = dim_date["full_date"].dt.year.astype(int)
dim_date["quarter"] = dim_date["full_date"].dt.quarter.astype(int)
dim_date["month"] = dim_date["full_date"].dt.month.astype(int)
dim_date["month_name"] = dim_date["full_date"].dt.strftime("%B")
dim_date["day"] = dim_date["full_date"].dt.day.astype(int)
dim_date["day_name"] = dim_date["full_date"].dt.strftime("%A")
dim_date["week_of_year"] = dim_date["full_date"].dt.isocalendar().week.astype(int)

dim_date = dim_date[[
    "date_key","full_date","year","quarter","month","month_name","day","day_name","week_of_year"
]]

dim_date.head()


,date_key,full_date,year,quarter,month,month_name,day,day_name,week_of_year
0,20101201,2010-12-01,2010,4,12,December,1,Wednesday,48
1,20101202,2010-12-02,2010,4,12,December,2,Thursday,48
2,20101203,2010-12-03,2010,4,12,December,3,Friday,48
3,20101205,2010-12-05,2010,4,12,December,5,Sunday,48
4,20101206,2010-12-06,2010,4,12,December,6,Monday,49


In [ ]:
dim_product = (
    sales[["StockCode", "Description"]]
    .drop_duplicates(subset=["StockCode"])
    .rename(columns={"StockCode": "stock_code", "Description": "description"})
    .reset_index(drop=True)
)

dim_product.head(20)


,stock_code,description
0,85123A,WHITE HANGING HEART T-LIGHT HOLDER
1,71053,WHITE METAL LANTERN
2,84406B,CREAM CUPID HEARTS COAT HANGER
3,84029G,KNITTED UNION FLAG HOT WATER BOTTLE
4,84029E,RED WOOLLY HOTTIE WHITE HEART.
5,22752,SET 7 BABUSHKA NESTING BOXES
6,21730,GLASS STAR FROSTED T-LIGHT HOLDER
7,22633,HAND WARMER UNION JACK
8,22632,HAND WARMER RED POLKA DOT
9,84879,ASSORTED COLOUR BIRD ORNAMENT


In [ ]:
# dim_customer (customer only)
dim_customer = (
    sales[["CustomerID"]]
    .drop_duplicates()
    .rename(columns={"CustomerID": "customer_code"})  # business key
    .reset_index(drop=True)
)

# dim_country (country only)
dim_country = (
    sales[["Country"]]
    .drop_duplicates()
    .rename(columns={"Country": "country_name"})
    .reset_index(drop=True)
)

# helpful flag for analysis
dim_country["is_uk"] = (dim_country["country_name"] == "United Kingdom").astype(int)

dim_customer.head(), dim_country.head()

(   customer_code
 0       17850.00
 1       13047.00
 2       12583.00
 3       13748.00
 4       15100.00,
      country_name  is_uk
 0  United Kingdom      1
 1          France      0
 2       Australia      0
 3     Netherlands      0
 4         Germany      0)

In [ ]:
fact = sales.copy()

# Create line number per invoice (unique line key within an invoice)
fact["invoice_line_no"] = fact.groupby("InvoiceNo").cumcount() + 1

# date_key for joins
fact["date_key"] = pd.to_datetime(fact["InvoiceDate"]).dt.strftime("%Y%m%d").astype(int)

# revenue (since sales = net dataset)
fact["revenue"] = fact["Quantity"] * fact["UnitPrice"]

# rename columns to SQL-friendly names
fact_sales = fact.rename(columns={
    "InvoiceNo": "invoice_no",
    "InvoiceDate": "invoice_datetime",
    "StockCode": "stock_code",
    "CustomerID": "customer_code",
    "Country": "country_name",
    "Quantity": "quantity",
    "UnitPrice": "unit_price"
})

# FACT = keys + measures only (no description text)
fact_sales = fact_sales[[
    "invoice_no", "invoice_line_no", "date_key", "invoice_datetime",
    "customer_code", "country_name",
    "stock_code",
    "quantity", "unit_price", "revenue"
]]

fact_sales.head()

,invoice_no,invoice_line_no,date_key,invoice_datetime,customer_code,country_name,stock_code,quantity,unit_price,revenue
0,536365,1,20101201,2010-12-01 08:26:00,17850.00,United Kingdom,85123A,6,2.55,15.30
1,536365,2,20101201,2010-12-01 08:26:00,17850.00,United Kingdom,71053,6,3.39,20.34
2,536365,3,20101201,2010-12-01 08:26:00,17850.00,United Kingdom,84406B,8,2.75,22.00
3,536365,4,20101201,2010-12-01 08:26:00,17850.00,United Kingdom,84029G,6,3.39,20.34
4,536365,5,20101201,2010-12-01 08:26:00,17850.00,United Kingdom,84029E,6,3.39,20.34


In [ ]:
print("dim_date:", dim_date.shape)
print("dim_product:", dim_product.shape)
print("dim_customer:", dim_customer.shape)
print("dim_country:", dim_country.shape)
print("fact_sales:", fact_sales.shape)

print("fact unique invoice lines duplicates:",
      fact_sales[["invoice_no","invoice_line_no"]].duplicated().sum())
print("revenue <= 0:", (fact_sales["revenue"] <= 0).sum())

dim_date: (305, 9)
dim_product: (3922, 2)
dim_customer: (4339, 1)
dim_country: (38, 2)
fact_sales: (530104, 10)
fact unique invoice lines duplicates: 0
revenue <= 0: 0


In [ ]:
dim_date.to_csv("dim_date.csv", index=False, encoding="utf-8")
dim_product.to_csv("dim_product.csv", index=False, encoding="utf-8")
dim_customer.to_csv("dim_customer.csv", index=False, encoding="utf-8")
dim_country.to_csv("dim_country.csv", index=False, encoding="utf-8")
fact_sales.to_csv("fact_sales.csv", index=False, encoding="utf-8")